# Stage 2 - judge MAIN (Kaggle **GPU T4 x2 required**, Internet ON)
Add the generate notebook's output (gen_main_seed0.zip and gen_main_seed1.zip) as input. 2 seeds x 8 files x 300 = 4800 judgments at about 2.4 s each = about 3.2 h (extrapolated from the pilot). Exports verdict files plus summary_main.csv and criteria.json (no responses).

In [ ]:
import os, glob, subprocess, torch
assert torch.cuda.is_available(), "NO GPU. Settings -> Accelerator -> GPU T4 x2, then restart. (The last attempt ran on CPU and timed out after 12 h.)"
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
TAG = "main"
GEN_ZIPS = sorted(glob.glob("/kaggle/input/**/gen_main_seed*.zip", recursive=True)); print(GEN_ZIPS); assert len(GEN_ZIPS) >= 1, "add the gen_main zips (generate notebook output) as input"
JUDGE_HF = "Qwen/Qwen2.5-7B-Instruct"; JUDGE_ID = "qwen25-7b"      # fp16 split over 2 GPUs; if memory errors: "Qwen/Qwen2.5-3B-Instruct" and JUDGE_ID = "qwen25-3b"
!git clone -q https://github.com/sarrazer24/reins-mini /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!mkdir -p runs
for z in GEN_ZIPS: subprocess.run(f"unzip -qo {z} -d runs", shell=True)
!pip install -q -U transformers accelerate
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS

In [ ]:
for d in sorted(glob.glob(f"runs/{TAG}/seed*")):
    print("judging", d, flush=True)
    subprocess.run(f"python -u scripts/11_run_judge.py --run-dir {d} --guise-dir /tmp/REINS/guise --judge-id {JUDGE_ID} --hf-model {JUDGE_HF}", shell=True, check=True)

In [ ]:
# outcome table next to the paper + the pre-registered criteria
subprocess.run(f"python scripts/12_evaluate.py --runs runs/{TAG} --judge-id {JUDGE_ID} --size 2B --out results/summary_{TAG}.csv", shell=True)

In [ ]:
!find runs/main -name "*.judged_*.jsonl" -o -name "gate_scores_*.csv" | zip -q judged_{JUDGE_ID}_main.zip -@
!cp results/summary_main.csv results/criteria.json . 2>/dev/null; ls -la judged_*.zip summary_main.csv criteria.json
!rm -rf /tmp/REINS